# **Volkswagen Maintenance Cost Prediction Model**

This notebook builds a machine learning model to predict the total maintenance cost for Volkswagen vehicles for the upcoming month. It uses a Random Forest Regressor trained on historical data that has been cleaned and preprocessed by the `06_data_processing_pipeline.ipynb` notebook.

The final output is a single-value prediction of the expected cost for the next calendar month.

## **Prerequisites: The Processed Dataset**

This model's accuracy is entirely dependent on the clean, structured data generated by the `06_data_processing_pipeline.ipynb` notebook. It specifically requires the `SERVICE_ORDER_BASE_clean.xlsx` file as its input.

The data from the pipeline is essential because it is:

- **Cleaned:** Unnecessary columns have been removed.

- **Standardized:** Dates and column names are in a consistent format.

- **Numerical:** All categorical features (like Asset, Model, and Product codes) have been converted to numerical representations using Label Encoding.

- **Robust:** Financial outliers in GRAND TOTAL, UNIT VALUE, and PRODUCT QUANTITY have been treated using the IQR method, preventing extreme values from skewing the model.

## **Import libraries**

Import necessary libraries.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

## **Load Processed Data**

The pipeline starts by loading the clean `SERVICE_ORDER_BASE_clean.xlsx` dataset.

In [ ]:
# Load outlier-treated data
try:
  df = pd.read_excel("data/SERVICE_ORDER_BASE_clean.xlsx")
  print(f"Total records loaded: {len(df):,}")
except FileNotFoundError:
  print("Error: Dataset file not found. Please run the data processing pipeline first.")

## **Isolate Volkswagen Data**

It then filters this data to include records only for Volkswagen vehicles.

**Note:** The model identifies 'Volkswagen' by selecting the second most frequent manufacturer code (`MANUFACTURER CODE`) in the dataset. This assumption is based on initial data exploration and should be validated if the underlying data distribution changes.

In [ ]:
# Find Volkswagen vehicles
manufacturer_counts = df['MANUFACTURER CODE'].value_counts()
volkswagen_code = manufacturer_counts.index[1]  # Second largest manufacturer
print(f"Using Manufacturer Code {volkswagen_code} as Volkswagen")

# Filter for Volkswagen vehicles
vw_data = df[df['MANUFACTURER CODE'] == volkswagen_code]
print(f"Volkswagen records: {len(vw_data):,} ({len(vw_data)/len(df)*100:.1f}% of total)")
print(f"Volkswagen vehicles: {vw_data['ASSET_CODE_encoded'].nunique():,}")

## **Aggregate Data Monthly**

### The individual, transaction-level service records are aggregated into a monthly time series.

This step calculates three key metrics for each month:

- **total_cost:** The sum of GRAND TOTAL for all services in that month.

- **vehicles:** The count of unique vehicles (`ASSET_CODE_encoded`) serviced.

- **preventive_ratio:** The ratio of preventive vs. corrective maintenance jobs.

In [ ]:
# Create monthly aggregations
monthly_vw = vw_data.groupby(['SERVICE_ORDER_year', 'SERVICE_ORDER_month']).agg({
  'GRAND TOTAL': 'sum',
  'ASSET_CODE_encoded': 'nunique',
  'PREVENTIVE_CORRECTIVE MAINTENANCE': 'mean'
}).reset_index()

monthly_vw.columns = ['year', 'month', 'total_cost', 'vehicles', 'preventive_ratio']
monthly_vw = monthly_vw.sort_values(['year', 'month']).reset_index(drop=True)

print(f"Monthly data points: {len(monthly_vw)}")
print(f"Date range: {monthly_vw['year'].min():.0f}-{monthly_vw['year'].max():.0f}")
print(f"Monthly cost range: R${monthly_vw['total_cost'].min():,.0f} - R${monthly_vw['total_cost'].max():,.0f}")

# Show cost statistics
print(f"\nCost statistics:")
print(f"  Mean monthly cost: R${monthly_vw['total_cost'].mean():,.0f}")
print(f"  Median monthly cost: R${monthly_vw['total_cost'].median():,.0f}")
print(f"  Standard deviation: R${monthly_vw['total_cost'].std():,.0f}")

## **Feature Engineering for Time Series**

To enable the model to learn from past trends, 7 predictive features are engineered from the monthly data. These include lag features (the cost from 1 and 2 months ago) and moving averages (the average cost over the last 3 months).

In [ ]:
# Create prediction features
monthly_vw['month_of_year'] = monthly_vw['month']
monthly_vw['cost_lag_1'] = monthly_vw['total_cost'].shift(1)
monthly_vw['cost_lag_2'] = monthly_vw['total_cost'].shift(2)
monthly_vw['cost_ma_3'] = monthly_vw['total_cost'].rolling(3).mean()
monthly_vw['vehicles_lag_1'] = monthly_vw['vehicles'].shift(1)

# Remove incomplete rows
model_data = monthly_vw.dropna()
print(f"\nModel training data: {len(model_data)} months")

# Prepare features and target
features = ['month_of_year', 'vehicles', 'preventive_ratio', 'cost_lag_1', 'cost_lag_2', 'cost_ma_3', 'vehicles_lag_1']
X = model_data[features]
y = model_data['total_cost']

## **Train the Prediction Model**

A `RandomForestRegressor` is used for its robustness and ability to capture non-linear relationships. The model is trained on the entire historical dataset to maximize its ability to capture all trends and seasonality, optimizing it for making future predictions.

In [ ]:
print(f"Training on all {len(X)} months of data")

# Train Random Forest model on all data
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Show feature importance
importance = pd.DataFrame({
  'feature': features,
  'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
for _, row in importance.iterrows():
  print(f"  {row['feature']}: {row['importance']:.3f}")

## **Predict Next Month's Cost**

Finally, the model uses the features from the most recent month of data to forecast the total maintenance cost for the next month.

In [ ]:
# Predict next month
latest_data = model_data.iloc[-1]
next_features = latest_data[features].values.reshape(1, -1)
predicted_cost = model.predict(next_features)[0]

# Get next month details
current_year = int(latest_data['year'])
current_month = int(latest_data['month'])
next_month = current_month + 1 if current_month < 12 else 1
next_year = current_year if current_month < 12 else current_year + 1

print(f"\nPrediction for {next_year}-{next_month:02d}:")
print(f"Predicted monthly cost: R${predicted_cost:,.0f}")
print(f"Historical average: R${monthly_vw['total_cost'].mean():,.0f}")
print(f"Recent 6-month average: R${monthly_vw['total_cost'].tail(6).mean():,.0f}")

## **Model Configuration**

The model is configured to train on all available historical data to make the most accurate prediction for the next month.

**Training Approach:**
- Uses all historical Volkswagen maintenance data
- No train/test split - optimized for production prediction
- Incorporates all patterns and seasonality from the complete dataset

## **Model Features Explained**
The model uses 7 engineered features to make its prediction:

- **month_of_year**: Captures seasonal patterns in maintenance needs (e.g., more AC repairs in summer).

- **vehicles**: The number of unique Volkswagen vehicles serviced in a given month. A primary driver of cost.

- **preventive_ratio**: The proportion of maintenance that was preventive. A higher ratio might indicate lower overall costs over time.

- **cost_lag_1**: The total maintenance cost from the previous month. Captures recent momentum.

- **cost_lag_2**: The total maintenance cost from two months ago. Helps identify broader, short-term trends.

- **cost_ma_3**: The 3-month moving average of costs. Smooths out short-term fluctuations to provide a more stable trend indicator.

- **vehicles_lag_1**: The number of unique vehicles serviced in the previous month.

### **Feature Importance Rankings** (from model analysis):
- **vehicles:** 0.727 (most important)
- **cost_lag_1:** 0.097
- **cost_ma_3:** 0.061
- **vehicles_lag_1:** 0.046
- **preventive_ratio:** 0.035
- **cost_lag_2:** 0.021
- **month_of_year:** 0.012 (least important)

### **Key Cost Drivers: Feature Importance**
The trained Random Forest model also allows us to understand what factors are most influential in predicting future costs. The feature importance rankings show:

- **vehicles:** The number of active vehicles is, by far, the most significant cost driver.

- **cost_lag_1** and **cost_ma_3**: Recent historical costs are strong indicators of near-future costs.

- **preventive_ratio** and **month_of_year**: The type of maintenance and seasonality have a smaller but still relevant impact.